# House Prices with Pairwise Difference Regression: raw DataFrame in

**Pairwise Difference Learning (PDL)** trains a model on *pairs* of rows: "do these two rows have the same label?" (or, for regression, "how far apart are their targets?"). A new row is then compared with the training rows, whose labels are known, and the answers are combined. With n training rows the model can learn from up to n² pairs, which is why it is aimed at small tables. [`pdll`](https://github.com/Karim-53/pdll) wraps any scikit-learn model with the usual `fit` / `predict` API. Method: [Belaid, Rabus & Hüllermeier, *Pairwise Difference Learning for Classification*, Discovery Science 2024](https://doi.org/10.1007/978-3-031-78980-9_18). *Disclosure: I am one of the authors.*

In [ ]:
!pip install -q -U pdll

In [ ]:
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score

import pdll
from pdll import PairwiseDifferenceRegressor

DATA_DIR = os.environ.get('DATA_DIR', '/kaggle/input/house-prices-advanced-regression-techniques')
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
print('pdll', pdll.__version__, '| train', train.shape, '| test', test.shape)

All 79 columns as they come, strings and missing values included. The competition scores the RMSE of log(SalePrice), so we model the log price directly.

In [ ]:
both = pd.concat([train, test]).drop(columns=['Id', 'SalePrice'])
for column in both.select_dtypes(exclude='number').columns:   # strings -> categories (same codes in train and test)
    both[column] = both[column].astype('category')
X, X_test = both.iloc[:len(train)], both.iloc[len(train):]
y = np.log1p(train['SalePrice'])
print(f'{X.shape[1]} columns, {sum(X.dtypes == "category")} categorical, {int(X.isna().sum().sum())} missing values')

In [ ]:
base = HistGradientBoostingRegressor(categorical_features='from_dtype', random_state=0)
pdl = PairwiseDifferenceRegressor(base, random_state=0)

cv = KFold(n_splits=5, shuffle=True, random_state=0)
for name, model in [('HistGradientBoosting', base), ('PDL(HistGradientBoosting)', pdl)]:
    rmse = -cross_val_score(model, X, y, cv=cv, scoring='neg_root_mean_squared_error')
    print(f'{name:28} RMSE(log price) {rmse.mean():.4f} ± {rmse.std():.4f}')

In [ ]:
pdl.fit(X, y)
submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': np.expm1(pdl.predict(X_test))})
submission.to_csv('submission.csv', index=False)
submission.head()

If PDL is useful in your work, please cite the paper:

```bibtex
@inproceedings{belaid2024pairwise,
  title     = {Pairwise Difference Learning for Classification},
  author    = {Belaid, Mohamed Karim and Rabus, Maximilian and H{\"u}llermeier, Eyke},
  booktitle = {Discovery Science},
  pages     = {284--299},
  publisher = {Springer},
  year      = {2024},
  doi       = {10.1007/978-3-031-78980-9_18}
}
```